1. Load and Initial Inspection

In [1]:
import pandas as pd 
import numpy as np

In [2]:
from pathlib import Path

base_dir = Path(".").resolve().parent
file_path = base_dir / "clean_data" / "hotel_booking_cleaned.csv"

df = pd.read_csv(file_path)
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_nights,total_guests,revenue
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,Transient,0.0,0,0,Check-Out,2015-07-01,0,2,0.0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,Transient,0.0,0,0,Check-Out,2015-07-01,0,2,0.0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,Transient,75.0,0,0,Check-Out,2015-07-02,1,1,75.0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,Transient,75.0,0,0,Check-Out,2015-07-02,1,1,75.0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,Transient,98.0,0,1,Check-Out,2015-07-03,2,2,196.0


In [3]:
print(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn dtypes:")
print(df.dtypes)

Shape  : 119,389 rows × 35 columns

Column dtypes:
hotel                              object
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                 object
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                            int64
babies                              int64
meal                               object
country                            object
market_segment                     object
distribution_channel               object
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                 object
assigned_room_type                 object
booking_changes                     int64
deposit_type             

2. Data Normalization

2.1 dim_hotel

In [4]:
hotel_vals = sorted(df["hotel"].dropna().unique())

dim_hotel = pd.DataFrame({
    "hotel_id"  : range(1, len(hotel_vals) + 1),
    "hotel_name": hotel_vals,
})

dim_hotel

,hotel_id,hotel_name
0,1,City Hotel
1,2,Resort Hotel


2.2 dim_meal

In [5]:
MEAL_DETAILS = {
    "BB": {"name": "Bed & Breakfast", "description": "Breakfast only"},
    "FB": {"name": "Full Board", "description": "Breakfast, lunch & dinner"},
    "HB": {"name": "Half Board", "description": "Breakfast & one other meal"},
    "SC": {"name": "Self Catering", "description": "No meals included"},
    "Undefined" : {"name": "Undefined", "description": "No meal package or undefined"},
}

meal_vals = sorted(df["meal"].dropna().unique())

dim_meal = pd.DataFrame({
    "meal_id": meal_vals,
    "meal_name": [MEAL_DETAILS.get(m, {}).get("name", m) for m in meal_vals],
    "meal_description": [MEAL_DETAILS.get(m, {}).get("description", "Undefined") for m in meal_vals]
})

dim_meal

,meal_id,meal_name,meal_description
0,BB,Bed & Breakfast,Breakfast only
1,FB,Full Board,"Breakfast, lunch & dinner"
2,HB,Half Board,Breakfast & one other meal
3,SC,Self Catering,No meals included
4,Undefined,Undefined,No meal package or undefined


2.3 dim_marketsegment

In [6]:
seg_vals = sorted(df["market_segment"].dropna().unique())
if "Undefined" in seg_vals:
    seg_vals.remove("Undefined")

segment_mapping = {"Undefined": 0}
for i, val in enumerate(seg_vals, start=1):
    segment_mapping[val] = i

all_segs = ["Undefined"] + seg_vals 

dim_market_segment = pd.DataFrame({
    "market_segment_id"   : [segment_mapping[seg] for seg in all_segs],
    "market_segment_name" : all_segs
})

dim_market_segment

,market_segment_id,market_segment_name
0,0,Undefined
1,1,Aviation
2,2,Complementary
3,3,Corporate
4,4,Direct
5,5,Groups
6,6,Offline TA/TO
7,7,Online TA


2.4 dim_distributionchannel

In [7]:
channel_vals = sorted(df["distribution_channel"].dropna().unique())
if "Undefined" in channel_vals:
    channel_vals.remove("Undefined")

channel_mapping = {"Undefined": 0}
for i, val in enumerate(channel_vals, start=1):
    channel_mapping[val] = i

all_channels = ["Undefined"] + channel_vals

dim_distribution_channel = pd.DataFrame({
    "distribution_channel_id"  : [channel_mapping[channel] for channel in all_channels],
    "distribution_channel_name": all_channels
})

dim_distribution_channel

,distribution_channel_id,distribution_channel_name
0,0,Undefined
1,1,Corporate
2,2,Direct
3,3,GDS
4,4,TA/TO


2.5 dim_deposittype

In [8]:
deposit_vals = sorted(df["deposit_type"].dropna().unique())

dim_deposit_type = pd.DataFrame({
    "deposit_type_id"  : range(1, len(deposit_vals) + 1),
    "deposit_type_name": deposit_vals,
})

dim_deposit_type

,deposit_type_id,deposit_type_name
0,1,No Deposit
1,2,Non Refund
2,3,Refundable


2.6 dim_customertype

In [9]:
ctype_vals = df["customer_type"].dropna().value_counts().index.tolist()

dim_customer_type = pd.DataFrame({
    "customer_type_id"   : range(1, len(ctype_vals) + 1),
    "customer_type_name" : ctype_vals,
})

dim_customer_type

,customer_type_id,customer_type_name
0,1,Transient
1,2,Transient-Party
2,3,Contract
3,4,Group


2.7 dim_reservationstatus

In [10]:
custom_order = ["Check-Out", "Canceled", "No-Show"]

existing_vals = df["reservation_status"].dropna().unique()

status_vals = [status for status in custom_order if status in existing_vals]

dim_reservation_status = pd.DataFrame({
    "reservation_status_id"   : range(1, len(status_vals) + 1),
    "reservation_status_name" : status_vals,
})

dim_reservation_status

,reservation_status_id,reservation_status_name
0,1,Check-Out
1,2,Canceled
2,3,No-Show


2.8 dim_country

In [11]:
!pip install pycountry
import pycountry

In [12]:
country_vals = sorted(df["country"].dropna().unique())

def get_country_name(code):
    if code == "Unknown":
        return "Unknown"
    try:
        return pycountry.countries.get(alpha_3=code).name
    except (AttributeError, KeyError):
        return "Unknown"

dim_country = pd.DataFrame({
    "country_id": country_vals,
    "country_name": [get_country_name(c) for c in country_vals]
})

dim_country.head()

,country_id,country_name
0,ABW,Aruba
1,AGO,Angola
2,AIA,Anguilla
3,ALB,Albania
4,AND,Andorra


4. Build FK Lookup Maps

In [13]:
hotel_map     = dict(zip(dim_hotel["hotel_name"],                          dim_hotel["hotel_id"]))
meal_map      = dict(zip(dim_meal["meal_id"],                            dim_meal["meal_id"]))
market_map    = dict(zip(dim_market_segment["market_segment_name"],        dim_market_segment["market_segment_id"]))
channel_map   = dict(zip(dim_distribution_channel["distribution_channel_name"], dim_distribution_channel["distribution_channel_id"]))
deposit_map   = dict(zip(dim_deposit_type["deposit_type_name"],            dim_deposit_type["deposit_type_id"]))
ctype_map     = dict(zip(dim_customer_type["customer_type_name"],          dim_customer_type["customer_type_id"]))
status_map    = dict(zip(dim_reservation_status["reservation_status_name"],dim_reservation_status["reservation_status_id"]))
country_map   = dict(zip(dim_country["country_id"],                      dim_country["country_id"]))

print("All lookup maps ready.")

All lookup maps ready.


5. build fact_booking

In [14]:
fact = df.copy().reset_index(drop=True)

# ── Surrogate primary key ─────────────────────────────────────────────────
fact.insert(0, "booking_id", range(1, len(fact) + 1))

# ── Map dimension FKs ─────────────────────────────────────────────────────
fact["hotel_id"]                = fact["hotel"].map(hotel_map)
fact["market_segment_id"]       = fact["market_segment"].map(market_map)
fact["distribution_channel_id"] = fact["distribution_channel"].map(channel_map)
fact["deposit_type_id"]         = fact["deposit_type"].map(deposit_map)
fact["customer_type_id"]        = fact["customer_type"].map(ctype_map)
fact["reservation_status_id"]   = fact["reservation_status"].map(status_map)
fact["meal_id"]               = fact["meal"]
fact["country_id"]            = fact["country"]

# ── Date Transformation ───────────────────────────────────────────────────
month_mapping = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}

fact["arrival_date"] = pd.to_datetime(pd.DataFrame({
    "year": fact["arrival_date_year"],
    "month": fact["arrival_date_month"].map(month_mapping),
    "day": fact["arrival_date_day_of_month"]
}))

# ── Select only the fact columns ──────────────────────────────────────────
FACT_COLUMNS = [

    # 1. main information & time
    "booking_id",
    "hotel_id",
    "arrival_date",
    'arrival_date_week_number',
    "lead_time",
    "total_nights",
    "stays_in_weekend_nights",
    "stays_in_week_nights",

    # 2. guest, room, and service details
    "total_guests",
    "adults",
    "children",
    "babies",
    "reserved_room_type",
    "assigned_room_type",
    "meal_id",
    "required_car_parking_spaces",
    "total_of_special_requests",

    # 3. sales channel & marketing
    "country_id",
    "market_segment_id",
    "distribution_channel_id",
    "agent",
    "company",

    # 4. finance & payment policies
    "adr",
    "revenue",
    "deposit_type_id",
    "customer_type_id",

    # 5. guest behavior profile
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",

    # 6. reservation status
    "is_canceled",
    "reservation_status_id",
    "reservation_status_date"
]

fact_booking = fact[FACT_COLUMNS]

print(f"FACT_Booking shape: {fact_booking.shape}")
fact_booking.head(3)

FACT_Booking shape: (119389, 34)


,booking_id,hotel_id,arrival_date,arrival_date_week_number,lead_time,total_nights,stays_in_weekend_nights,stays_in_week_nights,total_guests,adults,...,deposit_type_id,customer_type_id,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,is_canceled,reservation_status_id,reservation_status_date
0,1,2,2015-07-01,27,342,0,0,0,2,2,...,1,1,0,0,0,3,0,0,1,2015-07-01
1,2,2,2015-07-01,27,737,0,0,0,2,2,...,1,1,0,0,0,4,0,0,1,2015-07-01
2,3,2,2015-07-01,27,7,1,0,1,1,1,...,1,1,0,0,0,0,0,0,1,2015-07-02


3. Relationship Validation

In [15]:
# ── Foreign Key Checks ──────────────────────────────
FK_CHECKS = {
    "hotel_id"                : dim_hotel["hotel_id"],
    "market_segment_id"       : dim_market_segment["market_segment_id"],
    "distribution_channel_id" : dim_distribution_channel["distribution_channel_id"],
    "deposit_type_id"         : dim_deposit_type["deposit_type_id"],
    "customer_type_id"        : dim_customer_type["customer_type_id"],
    "reservation_status_id"   : dim_reservation_status["reservation_status_id"],
    "meal_id"               : dim_meal["meal_id"],
    "country_id"            : dim_country["country_id"],
}

# ── validation checks ──────────────
all_ok = True
for fk_col, valid_keys in FK_CHECKS.items():
    fact_vals  = fact_booking[fk_col].dropna()
    bad        = ~fact_vals.isin(valid_keys)
    null_count = fact_booking[fk_col].isna().sum()
    status     = "✅" if bad.sum() == 0 else "❌"
    if bad.sum() > 0:
        all_ok = False
    print(f"{status}  {fk_col:<35} nulls={null_count:>6,}  unresolved={bad.sum():>6,}")

print()
print("Referential integrity:", "PASSED ✅" if all_ok else "FAILED ❌ — check unresolved rows above")

✅  hotel_id                            nulls=     0  unresolved=     0
✅  market_segment_id                   nulls=     0  unresolved=     0
✅  distribution_channel_id             nulls=     0  unresolved=     0
✅  deposit_type_id                     nulls=     0  unresolved=     0
✅  customer_type_id                    nulls=     0  unresolved=     0
✅  reservation_status_id               nulls=     0  unresolved=     0
✅  meal_id                             nulls=     0  unresolved=     0
✅  country_id                          nulls=     0  unresolved=     0

Referential integrity: PASSED ✅


In [16]:
# open full column view for final check
pd.set_option('display.max_columns', None)

# recall the fact table after all transformations and checks
print(f"FACT_Booking shape: {fact_booking.shape}")
fact_booking.head(3)

FACT_Booking shape: (119389, 34)


,booking_id,hotel_id,arrival_date,arrival_date_week_number,lead_time,total_nights,stays_in_weekend_nights,stays_in_week_nights,total_guests,adults,children,babies,reserved_room_type,assigned_room_type,meal_id,required_car_parking_spaces,total_of_special_requests,country_id,market_segment_id,distribution_channel_id,agent,company,adr,revenue,deposit_type_id,customer_type_id,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,is_canceled,reservation_status_id,reservation_status_date
0,1,2,2015-07-01,27,342,0,0,0,2,2,0,0,C,C,BB,0,0,PRT,4,2,0,0,0.0,0.0,1,1,0,0,0,3,0,0,1,2015-07-01
1,2,2,2015-07-01,27,737,0,0,0,2,2,0,0,C,C,BB,0,0,PRT,4,2,0,0,0.0,0.0,1,1,0,0,0,4,0,0,1,2015-07-01
2,3,2,2015-07-01,27,7,1,0,1,1,1,0,0,A,C,BB,0,0,GBR,4,2,0,0,75.0,75.0,1,1,0,0,0,0,0,0,1,2015-07-02


4. Export Normalized Dataset

In [17]:
import os
import subprocess
import sys
from pathlib import Path

# ── 1. pandas installation case ─────────────────────
try:
    import pandas as pd
except ImportError:
    print("Pandas not found. Installing pandas automatically...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd

# ── 2. output directory setup ───────────────────────────────────────────
OUTPUT_DIR = Path(".").resolve().parent / "clean_data" / "Normalized Clean Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 3. generate table list ─────────
TABLES = {
    # Fact
    "fact_booking": fact_booking,
    # Dimensions
    "dim_hotel": dim_hotel,
    "dim_meal": dim_meal,
    "dim_market_segment": dim_market_segment,
    "dim_distribution_channel": dim_distribution_channel,
    "dim_deposit_type": dim_deposit_type,
    "dim_customer_type": dim_customer_type,
    "dim_reservation_status": dim_reservation_status,
    "dim_country": dim_country,
}

# ── 4. export tables (with existing file check) ───────────────────────────
print(f"Exporting {len(TABLES)} tables to '{OUTPUT_DIR}/'\n")
print(f"{'Table':<35} {'Rows':>8}  {'Columns':>8}  Status / File")
print("-" * 95)

for table_name, df_out in TABLES.items():
    file_path = OUTPUT_DIR / f"{table_name}.csv"
    
    # check if file already exists
    if file_path.exists():
        print(f"⏭️  {table_name:<33} {'-':>8}  {'-':>8}  Skipped (File already exists)")
    else:
        df_out.to_csv(file_path, index=False)
        print(f"💾 {table_name:<33} {len(df_out):>8,}  {len(df_out.columns):>8}  {file_path.name} (Exported)")

print("\nAll processing completed successfully! 🚀")

Exporting 9 tables to 'C:\Users\zakym_xg1tqji\Documents\GitHub\Travel-Tourism-Hospitality-Project\clean_data\Normalized Clean Data/'

Table                                   Rows   Columns  Status / File
-----------------------------------------------------------------------------------------------
💾 fact_booking                       119,389        34  fact_booking.csv (Exported)
💾 dim_hotel                                2         2  dim_hotel.csv (Exported)
💾 dim_meal                                 5         3  dim_meal.csv (Exported)
💾 dim_market_segment                       8         2  dim_market_segment.csv (Exported)
💾 dim_distribution_channel                 5         2  dim_distribution_channel.csv (Exported)
💾 dim_deposit_type                         3         2  dim_deposit_type.csv (Exported)
💾 dim_customer_type                        4         2  dim_customer_type.csv (Exported)
💾 dim_reservation_status                   3         2  dim_reservation_status.csv (Exported